In [ ]:
# Visualisation : image augmentee | 12 derivations en couleur | superposition
import os, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Patch

%matplotlib inline

DATA_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main\data"
STEM      = "ECG_031_167_p0_aug"   # a modifier au besoin

LEAD_NAMES  = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
LEAD_COLORS = [(1,0,0),(0,0.8,0),(0,0.4,1),(1,0.8,0),(1,0,1),(0,1,1),
               (1,0.5,0),(0.6,0,1),(0,0.5,0),(0.55,0.55,0.55),(1,0.4,0.7),(0.2,0.6,1)]

img_path = os.path.join(DATA_ROOT, "output_augmentation", "images", f"{STEM}.webp")
mask_dir = os.path.join(DATA_ROOT, "output_augmentation", "masks", STEM)
img = np.array(Image.open(img_path).convert("RGB"))
H, W = img.shape[:2]

def _lead_channel(folder, lead, H, W):
    """Fusionne (max) tous les segments/rythme d'une derivation -> canal [H,W] dans [0,1]."""
    acc = np.zeros((H, W), np.float32)
    for f in glob.glob(os.path.join(folder, f"mask_signal_{lead}_*.png")):
        try:
            m = np.array(Image.open(f).convert("L"))
        except Exception:
            continue  # masque corrompu/tronque (run P2 interrompu) -> ignore
        if m.shape != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_AREA)
        acc = np.maximum(acc, m.astype(np.float32))
    return acc / 255.0

composite = np.zeros((H, W, 3), np.float32)
for lead, color in zip(LEAD_NAMES, LEAD_COLORS):
    ch = _lead_channel(mask_dir, lead, H, W) > 0.4
    for k in range(3):
        composite[..., k][ch] = color[k]

overlay = img.astype(np.float32) / 255.0
m_any = composite.sum(2) > 0
overlay[m_any] = 0.30 * overlay[m_any] + 0.70 * composite[m_any]

fig, axes = plt.subplots(1, 3, figsize=(26, 8))
axes[0].imshow(img);                 axes[0].set_title("Image augmentee");                              axes[0].axis("off")
axes[1].imshow(composite);           axes[1].set_title("12 derivations (1 couleur = 1 derivation)");     axes[1].axis("off")
axes[2].imshow(overlay.clip(0, 1));  axes[2].set_title("Superposition");                                axes[2].axis("off")
axes[1].legend(handles=[Patch(color=c, label=n) for n, c in zip(LEAD_NAMES, LEAD_COLORS)],
               loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import os
import torch
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" # Pour voir l'erreur exacte si ça plante
torch.backends.cudnn.benchmark = False   # Désactive l'auto-optimisation qui peut figer au début

In [ ]:
import os
import sys
import time
import json
import datetime
import argparse

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg") 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

# C'est ici qu'on ajoute les bibliothèques de deep learning
import segmentation_models_pytorch as smp
import albumentations as A
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
import torch

In [ ]:
import os
import torch
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class TrainConfig:
    """Configuration d'entrainement - SEGMENTATION MULTI-DERIVATIONS (12 canaux)."""

    # -- Chemins --
    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    mask_dir: str = ""
    output_dir: str = ""

    # -- Cible : les 12 derivations (canaux 2-13 du multiclass), construites a la volee
    #    depuis mask_signal_{LEAD}_*.png (fusion des segments + rythme par derivation).
    mask_type: str = "per_lead_12_derivations"   # libelle seulement (non utilise comme fichier)
    lead_names: list = field(default_factory=lambda: [
        "I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"])

    # -- Architecture --
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 12          # <-- 12 derivations (multi-label, sigmoid par canal)

    # -- Resolution --
    img_height: int = 1024
    img_width: int = 1024

    # -- Entrainement --
    batch_size: int = 4
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    num_workers: int = 0
    pin_memory: bool = True

    # -- Loss & Scheduler --
    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    early_stop_patience: int = 15

    # -- Split --
    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources: list = field(default_factory=lambda: ["ECG_033"])

    # -- Device --
    device: str = ""

    # -- Divers --
    seed: int = 42
    save_every_n_epochs: int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.mask_dir:
            self.mask_dir = os.path.join(self.project_root, "output_augmentation", "masks")
        if not self.output_dir:
            # dossier SEPARE pour ne pas ecraser le modele binaire (runs_signal)
            self.output_dir = os.path.join(self.project_root, "training", "runs_signal_multiclass")

        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU detecte : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - lent a 1024x1024.")

cfg = TrainConfig()


In [ ]:
"""
Dataset PyTorch : SEGMENTATION MULTI-DERIVATIONS (12 canaux).

Cible = les 12 derivations (I, II, III, aVR, aVL, aVF, V1-V6). Chaque canal est
construit en fusionnant (max) tous les masks mask_signal_{LEAD}_*.png de l'image
(les segments seg0..segN ET le rythme _extra d'une meme derivation -> meme canal).

C'est un probleme MULTI-LABEL (les traces se chevauchent), donc 1 canal sigmoid
par derivation -- pas de softmax.

FILTRE BUG SHUFFLE : on EXCLUT les images dont la verite terrain est corrompue
(masques fantomes empiles par l'ancien bug 'shuffle'). Signature = une derivation
qui possede >=2 segments de GRILLE (mask_signal_<LEAD>_segN). ~3.7% des images.
"""

import os
import glob
import numpy as np
import cv2
import torch
from PIL import Image
from torch.utils.data import Dataset
import albumentations as A

LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]


def load_lead_channel(mask_folder, lead, H, W):
    """Fusionne (max) segments + rythme d'une derivation -> canal [H,W] dans [0,1]."""
    acc = np.zeros((H, W), np.float32)
    for f in glob.glob(os.path.join(mask_folder, f"mask_signal_{lead}_*.png")):
        try:
            m = np.array(Image.open(f).convert("L"))
        except Exception:
            continue  # masque corrompu/tronque (run P2 interrompu) -> ignore
        if m.shape != (H, W):
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_AREA)
        acc = np.maximum(acc, m.astype(np.float32))
    return acc / 255.0


def build_multiclass_target(mask_folder, H, W, lead_names=None):
    """Retourne un mask (H, W, 12) : un canal par derivation."""
    lead_names = lead_names or LEAD_NAMES
    chans = [load_lead_channel(mask_folder, lead, H, W) for lead in lead_names]
    return np.stack(chans, axis=-1).astype(np.float32)


def mask_folder_is_corrupted(mask_folder, lead_names=None):
    """True si une derivation a >=2 masques de GRILLE (signature du bug 'shuffle').
       Ne charge aucun pixel : juste les noms de fichiers (rapide)."""
    lead_names = lead_names or LEAD_NAMES
    for lead in lead_names:
        if len(glob.glob(os.path.join(mask_folder, f"mask_signal_{lead}_seg*.png"))) >= 2:
            return True
    return False


class ECGGridDataset(Dataset):
    """Dataset multi-derivations. (Nom conserve pour compatibilite avec la boucle train.)

    Chaque sample = (image_augmentee [3,H,W], cible 12 derivations [12,H,W]).
    `mask_type` est accepte mais IGNORE (la cible est construite depuis les masks par derivation).
    `skip_corrupted=True` exclut les images a verite terrain corrompue (bug shuffle).
    """

    def __init__(
        self,
        image_dir: str,
        mask_dir: str,
        mask_type: str = None,          # ignore (compat)
        source_prefixes: list = None,
        img_height: int = 1024,
        img_width: int = 1024,
        augment: bool = False,
        lead_names: list = None,
        skip_corrupted: bool = True,    # <-- exclut les images shuffle corrompues
    ):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_height = img_height
        self.img_width = img_width
        self.lead_names = lead_names or LEAD_NAMES

        self.samples = []
        self.n_skipped_corrupted = 0
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue
            if source_prefixes and not any(fname.startswith(p) for p in source_prefixes):
                continue
            stem = fname.replace(".webp", "")
            folder = os.path.join(mask_dir, stem)
            if not os.path.isdir(folder):
                continue
            # il faut au moins un mask de signal
            if not glob.glob(os.path.join(folder, "mask_signal_*.png")):
                continue
            # EXCLURE les images dont la verite terrain est corrompue (bug shuffle)
            if skip_corrupted and mask_folder_is_corrupted(folder, self.lead_names):
                self.n_skipped_corrupted += 1
                continue
            self.samples.append({
                "image_path": os.path.join(image_dir, fname),
                "mask_folder": folder,
                "stem": stem,
            })

        # Augmentations supplementaires LEGERES (en plus de celles de P2).
        # Pas de VerticalFlip : un ECG tete-en-bas n'est pas realiste.
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} paires trouvees"
              f" (12 derivations, sources: {source_prefixes or 'toutes'})"
              f"  [exclues corrompues (shuffle): {self.n_skipped_corrupted}]")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        img = Image.open(s["image_path"]).convert("RGB")
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0

        mask_np = build_multiclass_target(
            s["mask_folder"], self.img_height, self.img_width, self.lead_names)  # (H,W,12)

        if self.transform:
            t = self.transform(image=img_np, mask=mask_np)
            img_np = t["image"]
            mask_np = t["mask"]

        img_tensor = torch.from_numpy(np.ascontiguousarray(img_np)).permute(2, 0, 1).float()   # [3,H,W]
        mask_tensor = torch.from_numpy(np.ascontiguousarray(mask_np)).permute(2, 0, 1).float()  # [12,H,W]
        return img_tensor, mask_tensor


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VERIFICATION : la cible est construite DIRECTEMENT depuis les PNG
# (data/output_augmentation/masks/<stem>/mask_signal_<DERIV>_*.png) -- aucun NPZ.
# Affiche, pour un echantillon, les 12 derivations chargees une par une.
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# <-- change l'echantillon a verifier ici
STEM = "ECG_031_148_p0_aug"

img_path    = os.path.join(cfg.image_dir, STEM + ".webp")
mask_folder = os.path.join(cfg.mask_dir, STEM)
print("Image :", img_path)
print("Masks :", mask_folder, "  (existe:", os.path.isdir(mask_folder), ")")
print("-" * 70)

img = np.array(Image.open(img_path).convert("RGB"))
H, W = img.shape[:2]

# Pour chaque derivation : lister SES png et fusionner (via la meme fonction que le dataset)
fig, axes = plt.subplots(3, 5, figsize=(26, 15))
axes = axes.ravel()
axes[0].imshow(img); axes[0].set_title("Image augmentee", fontsize=11); axes[0].axis("off")

for i, lead in enumerate(cfg.lead_names):
    files = sorted(glob.glob(os.path.join(mask_folder, f"mask_signal_{lead}_*.png")))
    ch = load_lead_channel(mask_folder, lead, H, W)   # <- fonction du dataset (cellule precedente)
    m = ch > 0.4
    print(f"{lead:4s}: {len(files)} png -> {[os.path.basename(f) for f in files]}  ({int(m.sum())} px)")

    ax = axes[i + 1]
    ax.imshow(img)
    overlay = np.zeros((H, W, 4), np.float32)
    overlay[m] = [1, 0, 0, 1]          # masque de CETTE derivation en rouge
    ax.imshow(overlay)
    ax.set_title(f"{lead}  ({len(files)} png, {int(m.sum())} px)", fontsize=11)
    ax.axis("off")

for j in range(len(cfg.lead_names) + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"VERIFICATION -- 12 derivations chargees DIRECTEMENT depuis les PNG  ({STEM})", fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
"""
Script d'entraînement U-Net pour la détection de grille ECG.

Usage:
    python training/train.py                    # Auto-détecte CPU/GPU
    python training/train.py --device cpu       # Force CPU
    python training/train.py --device cuda      # Force GPU
    python training/train.py --epochs 5         # Override rapide
    python training/train.py --mask mask_grid_combined.png  # Autre mask

Tout est loggé dans training/runs/<timestamp>/
"""

import os
import sys
import time
import argparse
import datetime
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")  # backend non-interactif
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

# Ajouter le projet au path
sys.path.insert(0, os.path.abspath(""))

# ── Multi-derivations : noms, couleurs, helpers (ajout multiclasse) ──
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
LEAD_COLORS = [(1,0,0),(0,0.8,0),(0,0.4,1),(1,0.8,0),(1,0,1),(0,1,1),
               (1,0.5,0),(0.6,0,1),(0,0.5,0),(0.55,0.55,0.55),(1,0.4,0.7),(0.2,0.6,1)]

def composite_from_channels(ch_stack, thr=0.5):
    """ch_stack [C,H,W] (proba ou binaire) -> image RGB, une couleur par derivation."""
    C, H, W = ch_stack.shape
    out = np.zeros((H, W, 3), np.float32)
    for k in range(min(C, len(LEAD_COLORS))):
        m = ch_stack[k] > thr
        for j in range(3):
            out[..., j][m] = LEAD_COLORS[k][j]
    return out

@torch.no_grad()
def per_lead_dice(model, loader, device, n_classes, thr=0.5):
    """Dice par derivation sur tout le loader."""
    model.eval()
    inter = torch.zeros(n_classes); psum = torch.zeros(n_classes); tsum = torch.zeros(n_classes)
    for images, masks in loader:
        images = images.to(device)
        pred = (torch.sigmoid(model(images)) > thr).float().cpu()
        tgt = (masks > thr).float()
        inter += (pred * tgt).sum(dim=(0, 2, 3))
        psum += pred.sum(dim=(0, 2, 3))
        tsum += tgt.sum(dim=(0, 2, 3))
    return ((2 * inter + 1e-6) / (psum + tsum + 1e-6)).numpy()


#from training.config import TrainConfig
#from training.dataset import ECGGridDataset


# ═══════════════════════════════════════════════════════════════════════
# Loss Functions
# ═══════════════════════════════════════════════════════════════════════

class DiceLoss(nn.Module):
    """Dice Loss pour segmentation binaire. Gère le déséquilibre de classes."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred_sig = torch.sigmoid(pred)
        intersection = (pred_sig * target).sum(dim=(2, 3))
        union = pred_sig.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class BCEDiceLoss(nn.Module):
    """Combinaison BCE + Dice (souvent la meilleure pour la segmentation)."""
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight

    def forward(self, pred, target):
        return (self.bce_weight * self.bce(pred, target)
                + (1 - self.bce_weight) * self.dice(pred, target))


def get_loss(loss_type, bce_weight=0.5):
    """Factory pour la loss function."""
    if loss_type == "bce":
        return nn.BCEWithLogitsLoss()
    elif loss_type == "dice":
        return DiceLoss()
    elif loss_type == "bce_dice":
        return BCEDiceLoss(bce_weight)
    else:
        raise ValueError(f"Loss inconnue: {loss_type}")


# ═══════════════════════════════════════════════════════════════════════
# Métriques
# ═══════════════════════════════════════════════════════════════════════

def compute_metrics(pred, target, threshold=0.5):
    """Calcule IoU et Dice sur un batch."""
    with torch.no_grad():
        pred_bin = (torch.sigmoid(pred) > threshold).float()

        intersection = (pred_bin * target).sum(dim=(2, 3))
        pred_sum = pred_bin.sum(dim=(2, 3))
        target_sum = target.sum(dim=(2, 3))

        # Dice
        dice = (2 * intersection + 1e-6) / (pred_sum + target_sum + 1e-6)

        # IoU (Jaccard)
        union = pred_sum + target_sum - intersection
        iou = (intersection + 1e-6) / (union + 1e-6)

        # Pixel accuracy (sur les pixels de grille seulement)
        # = recall / sensitivity
        recall = (intersection + 1e-6) / (target_sum + 1e-6)

        # Precision
        precision = (intersection + 1e-6) / (pred_sum + 1e-6)

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }


# ═══════════════════════════════════════════════════════════════════════
# Visualisation
# ═══════════════════════════════════════════════════════════════════════

def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    """[image | derivations vraies | derivations predites] en couleur."""
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        true_c = composite_from_channels(masks_true[i].cpu().numpy(), thr=0.5)
        pred_c = composite_from_channels(torch.sigmoid(masks_pred[i]).cpu().numpy(), thr=0.5)
        axes[i, 0].imshow(img);    axes[i, 0].set_title("Image", fontsize=9);                axes[i, 0].axis("off")
        axes[i, 1].imshow(true_c); axes[i, 1].set_title("Derivations vraies", fontsize=9);    axes[i, 1].axis("off")
        axes[i, 2].imshow(pred_c); axes[i, 2].set_title("Derivations predites", fontsize=9);  axes[i, 2].axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════
# Entraînement
# ═══════════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Entraîne un epoch, retourne la loss moyenne et les métriques."""
    model.train()
    total_loss = 0
    total_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    n_batches = 0

    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)

        # Forward
        pred = model(images)
        loss = criterion(pred, masks)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Métriques
        metrics = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in total_metrics:
            total_metrics[k] += metrics[k]
        n_batches += 1

        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{metrics['dice']:.3f}")

    avg_loss = total_loss / max(n_batches, 1)
    avg_metrics = {k: v / max(n_batches, 1) for k, v in total_metrics.items()}
    return avg_loss, avg_metrics


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Valide le modèle, retourne la loss moyenne et les métriques."""
    model.eval()
    total_loss = 0
    total_metrics = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    n_batches = 0

    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        pred = model(images)
        loss = criterion(pred, masks)

        metrics = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in total_metrics:
            total_metrics[k] += metrics[k]
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_metrics = {k: v / max(n_batches, 1) for k, v in total_metrics.items()}
    return avg_loss, avg_metrics


def train(cfg: TrainConfig, resume_from=None):
    """Boucle d'entraînement principale."""

    # ── Setup ────────────────────────────────────────────────────────
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    # Sauvegarder la config
    config_dict = {k: str(v) if not isinstance(v, (int, float, bool, list, type(None)))
                   else v for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(config_dict, f, indent=2)

    print(f"\n{'='*60}")
    print(f"  Entrainement U-Net -- Segmentation 12 derivations")
    print(f"{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Resolution  : {cfg.img_height}x{cfg.img_width}")
    print(f"  Mask cible  : {cfg.mask_type}")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}")
    print(f"{'='*60}\n")

    # Seed
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)

    device = torch.device(cfg.device)

    # ── Datasets ─────────────────────────────────────────────────────
    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  Masks:  {cfg.mask_dir}")

    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_dataset = ECGGridDataset(
        image_dir=cfg.image_dir,
        mask_dir=cfg.mask_dir,
        mask_type=cfg.mask_type,
        source_prefixes=cfg.train_sources,
        img_height=cfg.img_height,
        img_width=cfg.img_width,
        augment=True,
    )

    print(f"  Val (sources: {cfg.val_sources}):")
    val_dataset = ECGGridDataset(
        image_dir=cfg.image_dir,
        mask_dir=cfg.mask_dir,
        mask_type=cfg.mask_type,
        source_prefixes=cfg.val_sources,
        img_height=cfg.img_height,
        img_width=cfg.img_width,
        augment=False,
    )

    if len(train_dataset) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee !")
        print("   Vérifie que les images et masks existent dans les dossiers ci-dessus.")
        print("   Exécute d'abord les pipelines P1 et P2 pour générer les données.")
        return

    train_loader = DataLoader(
        train_dataset, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
    )

    # ── Modèle ───────────────────────────────────────────────────────
    print("\n[*] Construction du modele...")
    import segmentation_models_pytorch as smp

    model = smp.Unet(
        encoder_name=cfg.encoder_name,
        encoder_weights=cfg.encoder_weights,
        in_channels=cfg.in_channels,
        classes=cfg.num_classes,
        activation=None,  # on applique sigmoid dans la loss
    )
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parametres: {total_params:,} (trainable: {trainable:,})")

    # ── Loss, Optimizer, Scheduler ───────────────────────────────────
    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience,
        factor=cfg.scheduler_factor,
    )

    # ── Historique ───────────────────────────────────────────────────
    history = {
        "train_loss": [], "val_loss": [],
        "train_dice": [], "val_dice": [],
        "train_iou": [], "val_iou": [],
        "lr": [],
    }
    best_val_dice = 0
    epochs_no_improve = 0
    start_epoch = 1

    # -- Reprise depuis un checkpoint (resume) --
    if resume_from:
        ckpt = torch.load(resume_from, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val_dice = float(ckpt.get("val_dice", 0) or 0)
        print(f"[RESUME] reprise depuis epoch {ckpt['epoch']} (val_dice {best_val_dice:.4f}) -> demarre a l'epoch {start_epoch}")

    # ── Boucle d'entraînement ────────────────────────────────────────
    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()

    for epoch in range(start_epoch, cfg.num_epochs + 1):
        t_epoch = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        # Train
        train_loss, train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validation
        val_loss, val_metrics = validate(model, val_loader, criterion, device)

        # Scheduler
        scheduler.step(val_loss)

        # Historique
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_dice"].append(train_metrics["dice"])
        history["val_dice"].append(val_metrics["dice"])
        history["train_iou"].append(train_metrics["iou"])
        history["val_iou"].append(val_metrics["iou"])
        history["lr"].append(current_lr)

        # Affichage
        elapsed = time.time() - t_epoch
        print(
            f"Epoch {epoch:3d}/{cfg.num_epochs} | "
            f"Train Loss: {train_loss:.4f}  Dice: {train_metrics['dice']:.3f} | "
            f"Val Loss: {val_loss:.4f}  Dice: {val_metrics['dice']:.3f}  "
            f"IoU: {val_metrics['iou']:.3f} | "
            f"LR: {current_lr:.1e} | {elapsed:.1f}s"
        )

        # Sauvegarder le meilleur modèle
        if val_metrics["dice"] > best_val_dice:
            best_val_dice = val_metrics["dice"]
            epochs_no_improve = 0
            best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": best_val_dice,
                "config": config_dict,
            }, best_path)
            print(f"  [BEST] Nouveau meilleur modele sauvegarde (Dice: {best_val_dice:.4f})")
        else:
            epochs_no_improve += 1

        # Checkpoint périodique
        if epoch % cfg.save_every_n_epochs == 0:
            ckpt_path = os.path.join(
                run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"
            )
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_metrics["dice"],
            }, ckpt_path)

        # Visualisation périodique
        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                # Prendre le premier batch de validation
                sample_images, sample_masks = next(iter(val_loader))
                sample_pred = model(sample_images.to(device)).cpu()
                vis_path = os.path.join(
                    run_dir, "visualizations", f"epoch_{epoch:03d}.png"
                )
                save_prediction_grid(sample_images, sample_masks, sample_pred, vis_path)

        # Early stopping
        if epochs_no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration")
            break

    # ── Fin ───────────────────────────────────────────────────────────
    total_time = time.time() - t_start
    print(f"\n{'='*60}")
    print(f"  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}")
    print(f"{'='*60}")

    # Sauvegarder l'historique
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # Courbes d'entraînement
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))

    # Dice par derivation (diagnostic final sur la validation)
    final_pld = per_lead_dice(model, val_loader, device, cfg.num_classes)
    print("\n[Dice par derivation - validation finale]")
    for name, d in zip(cfg.lead_names, final_pld):
        print(f"   {name:4s} : {d:.3f}")
    history["val_per_lead_dice_final"] = [float(x) for x in final_pld]
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    return run_dir


def _plot_training_curves(history, save_path):
    """Trace les courbes de loss et Dice."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history["train_loss"]) + 1)

    # Loss
    ax1.plot(epochs, history["train_loss"], "b-", label="Train")
    ax1.plot(epochs, history["val_loss"], "r-", label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss (BCE + Dice)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Dice
    ax2.plot(epochs, history["train_dice"], "b-", label="Train Dice")
    ax2.plot(epochs, history["val_dice"], "r-", label="Val Dice")
    ax2.plot(epochs, history["train_iou"], "b--", alpha=0.5, label="Train IoU")
    ax2.plot(epochs, history["val_iou"], "r--", alpha=0.5, label="Val IoU")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.set_title("Dice & IoU")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [PLOT] Courbes sauvegardees: {save_path}")


# ═══════════════════════════════════════════════════════════════════════
# CLI
# ═══════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser(
        description="Entraîner un U-Net pour la détection de grille ECG"
    )
    parser.add_argument("--device", type=str, default="",
                        help="Device: 'cpu' ou 'cuda' (auto si vide)")
    parser.add_argument("--epochs", type=int, default=None,
                        help="Nombre d'epochs (override)")
    parser.add_argument("--batch-size", type=int, default=None,
                        help="Taille du batch (override)")
    parser.add_argument("--lr", type=float, default=None,
                        help="Learning rate (override)")
    parser.add_argument("--resolution", type=int, default=None,
                        help="Résolution carrée (ex: 512)")
    parser.add_argument("--mask", type=str, default=None,
                        help="Type de mask (ex: mask_grid_major.png)")
    parser.add_argument("--encoder", type=str, default=None,
                        help="Encoder (ex: resnet34, resnet50)")
    args, unknown = parser.parse_known_args()

    # Construire la config avec les overrides
    overrides = {}
    if args.device:
        overrides["device"] = args.device
    if args.mask:
        overrides["mask_type"] = args.mask

    cfg = TrainConfig(**overrides)

    # Overrides post-init (après l'ajustement CPU/GPU automatique)
    if args.epochs is not None:
        cfg.num_epochs = args.epochs
    if args.batch_size is not None:
        cfg.batch_size = args.batch_size
    if args.lr is not None:
        cfg.learning_rate = args.lr
    if args.resolution is not None:
        cfg.img_height = args.resolution
        cfg.img_width = args.resolution
    if args.encoder is not None:
        cfg.encoder_name = args.encoder

    train(cfg)


# Pour REPRENDRE un entrainement interrompu : mets le chemin d'un checkpoint.
# Laisse None pour un entrainement NEUF.
RESUME_FROM = None
# RESUME_FROM = "data/training/runs_signal_multiclass/run_20260616_161515/checkpoints/checkpoint_epoch010.pth"

if __name__ == "__main__":
    if RESUME_FROM:
        train(cfg, resume_from=RESUME_FROM)
    else:
        main()


In [ ]:
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import segmentation_models_pytorch as smp
from IPython.display import display

# Trouver automatiquement le dernier run
runs_dir = cfg.output_dir  # /content/drive/MyDrive/data/training/runs

run_dirs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"❌ Aucun run trouvé dans {runs_dir}")

run_dir = run_dirs[-1]  # tri alphabétique = tri chronologique
print(f"✅ Run sélectionné : {run_dir}")

# Priorité : best_model > dernier checkpoint périodique
best_model_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
all_checkpoints = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))

if os.path.exists(best_model_path):
    checkpoint_path = best_model_path
    print(f"🏆 Meilleur modèle : {checkpoint_path}")
elif all_checkpoints:
    checkpoint_path = all_checkpoints[-1]
    print(f"📦 Dernier checkpoint : {checkpoint_path}")
else:
    raise FileNotFoundError(f"❌ Aucun checkpoint trouvé dans {run_dir}/checkpoints/")

# Charger le modèle
DEVICE = cfg.device

model = smp.Unet(
    encoder_name=cfg.encoder_name,
    encoder_weights=None,
    in_channels=cfg.in_channels,
    classes=cfg.num_classes,
)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()
print(f"✅ Modèle chargé — epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")

In [ ]:
image_dir = cfg.image_dir

val_images   = sorted([f for f in os.listdir(image_dir) if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir) if (f.startswith("ECG_031") or f.startswith("ECG_032")) and f.endswith(".webp")])



In [ ]:
%matplotlib inline
SET   = "val"   # "val" ou "train"
INDEX = 200
    # index dans la liste

image_list = val_images if SET == "val" else train_images
if INDEX >= len(image_list):
    raise IndexError(f"INDEX={INDEX} hors limites - {len(image_list)} images dans '{SET}'")

sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
mask_folder = os.path.join(cfg.mask_dir, sample_file.replace(".webp", ""))
print(f"Image : {sample_file}")
print(f"Masks : {mask_folder}  (existe: {os.path.isdir(mask_folder)})")

# Image
img_pil = Image.open(sample_path).convert("RGB").resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np  = np.array(img_pil, dtype=np.float32) / 255.0
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)

# Cible 12 derivations + prediction 12 derivations
target = build_multiclass_target(mask_folder, cfg.img_height, cfg.img_width, cfg.lead_names)  # (H,W,12)
with torch.no_grad():
    pred = torch.sigmoid(model(img_tensor)).squeeze(0).cpu().numpy()   # (12,H,W)

tgt_chw = target.transpose(2, 0, 1)            # (12,H,W)
pb = (pred > 0.5).astype(np.float32)
tb = (tgt_chw > 0.5).astype(np.float32)

# Dice par derivation
print("\n[Dice par derivation sur cette image]")
dices = []
for k, name in enumerate(cfg.lead_names):
    inter = (pb[k] * tb[k]).sum()
    d = (2 * inter) / (pb[k].sum() + tb[k].sum() + 1e-8)
    dices.append(d)
    print(f"   {name:4s} : {d:.3f}")
print(f"   {'MOY':4s} : {np.mean(dices):.3f}")

true_c = composite_from_channels(tgt_chw, thr=0.5)
pred_c = composite_from_channels(pred,    thr=0.5)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle(f"[{SET.upper()}]  {sample_file}  -  Dice moyen: {np.mean(dices):.3f}", fontsize=12)
axes[0].imshow(img_np); axes[0].set_title("Image originale");                      axes[0].axis("off")
axes[1].imshow(true_c); axes[1].set_title("Derivations vraies (couleur)");          axes[1].axis("off")
axes[2].imshow(pred_c); axes[2].set_title("Derivations predites (couleur)");        axes[2].axis("off")
from matplotlib.patches import Patch
axes[2].legend(handles=[Patch(color=c, label=n) for n, c in zip(LEAD_NAMES, LEAD_COLORS)],
               loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show(); plt.close()


In [ ]:
# Test du modele 12-derivations sur image reelle (data/output_real/)
import os, glob, cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Patch
import segmentation_models_pytorch as smp

%matplotlib inline

# === A modifier au besoin ===
REAL_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main\data\output_real"
SUBFOLDER = "augmentation_rotation_15"   # ou None pour scanner tout
INDEX     = 44

if SUBFOLDER:
    search = os.path.join(REAL_ROOT, SUBFOLDER, "*")
else:
    search = os.path.join(REAL_ROOT, "**", "*")
real_files = sorted([f for f in glob.glob(search, recursive=True)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))])
print(f"{len(real_files)} images reelles trouvees")
if INDEX >= len(real_files):
    raise IndexError(f"INDEX={INDEX} hors limites (max {len(real_files)-1})")
img_path = real_files[INDEX]
print(f"Image : {img_path}")

# Modele (si pas deja en memoire)
try:
    model
except NameError:
    runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
    ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                     in_channels=cfg.in_channels, classes=cfg.num_classes)
    ckpt = torch.load(ckpt_path, map_location=cfg.device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(cfg.device).eval()
    print(f"Modele charge - epoch {ckpt['epoch']} | val_dice = {ckpt.get('val_dice', 'N/A')}")

# Preprocessing (identique au training : /255)
img_pil = Image.open(img_path).convert("RGB")
W_orig, H_orig = img_pil.size
img_resized = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np = np.array(img_resized, dtype=np.float32) / 255.0
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(cfg.device)

# Inference 12 derivations
with torch.no_grad():
    pred = torch.sigmoid(model(img_tensor)).squeeze(0).cpu().numpy()   # (12,h,w)

# Remettre a la resolution native + composite couleur
pred_full = np.stack([cv2.resize(pred[k], (W_orig, H_orig), interpolation=cv2.INTER_LINEAR)
                      for k in range(pred.shape[0])], axis=0)
pred_c = composite_from_channels(pred_full, thr=0.5)

img_arr = np.array(img_pil).astype(np.float32) / 255.0
overlay = img_arr.copy()
m_any = pred_c.sum(2) > 0
overlay[m_any] = 0.30 * overlay[m_any] + 0.70 * pred_c[m_any]

print(f"Pixels signal predits : {int((pred_full > 0.5).any(0).sum())} "
      f"({100*(pred_full > 0.5).any(0).mean():.2f}%)")

fig, axes = plt.subplots(1, 3, figsize=(26, 9))
fig.suptitle(f"Inference 12 derivations : {os.path.basename(img_path)}", fontsize=12)
axes[0].imshow(np.array(img_pil)); axes[0].set_title(f"Image reelle ({W_orig}x{H_orig})"); axes[0].axis("off")
axes[1].imshow(pred_c);            axes[1].set_title("Derivations predites (couleur)");      axes[1].axis("off")
axes[2].imshow(overlay.clip(0, 1)); axes[2].set_title("Superposition");                      axes[2].axis("off")
axes[1].legend(handles=[Patch(color=c, label=n) for n, c in zip(LEAD_NAMES, LEAD_COLORS)],
               loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
plt.tight_layout(); plt.show()
